# ② 지식 추출 확대: 담당자별 역할표 행 38개 (검수 직전까지)

시나리오 표 4행(11/4 오전)에서만 뽑았던 학교 행동을, **본문의 담당자별 역할표**로 넓힌다. 같은 스키마·프롬프트(`school_action_v1`)·자동 검사를 쓴다(코드: `src/preprocess/school/extract.py`).

| 문서 | 표 | 행 수 | 단계 |
|---|---|---|---|
| 교육부 | 시나리오 대응 2단계(p38, 교육청 행 제외) | 3 | 대응 제2단계 |
| 교육부 | [표 3-9] 능동감시 실시 체계(p60) | 3 | 제2단계 |
| 교육부 | [표 3-11]~[표 3-16] 담당자별 역할(p65~67) | 15 | 제3단계(본문 절 경로로 확인) |
| 경기도 | 시나리오 단계 2(p56, 교육청 행 제외) | 3 | 제2단계 |
| 경기도 | 주요 구성원의 주요 역할(p165~168) | 14 | 행의 '단계' 칸 |

**일부러 뺀 표:** 표 3-6(일시적 관찰실 운영방안)·표 3-8(능동감시 운영방안)은 '속성이 행'이라 담당자 행동이 아니고, 표 3-18(복구단계)은 열 순서가 달라 별도 처리가 필요하다. 교육청 행은 학교 밖 기관의 업무다.

- 표 제목에 단계가 없는 표는 **본문 절 경로**(예: `4. 대응 제3단계 > 나. 관련 조치사항 내용`)를 '편집자 참고 메모'로만 모델에 알려 준다(원문 인용 아님).
- 쪽 대조는 사용자가 11/4에 선정 쪽수 전체를 확인한 것으로 보고 `VERIFIED_PDF`로 둔다.
- **API 비용:** `EXTRACTED_RUN = None`이면 LLM을 38번 호출한다. 폴더 이름을 넣으면 호출하지 않는다.
- 결과는 Git 제외 `data/interim/school/run_*`에 저장되고 모든 후보는 `PENDING`이다.

In [1]:
import os
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

from index.school.dictionaries import load_dictionaries
from index.school.id_map import load_manifest
from index.school.normalize import load_artifacts, load_raw_texts, precheck, read_jsonl
from preprocess.school.chunks import build_all_chunks
from preprocess.school.extract import (
    DEFAULT_MODEL,
    annotate_candidates,
    build_role_row_chunks,
    extract_chunk,
    load_concept_draft,
    make_model,
    save_run,
)
from preprocess.school.markdown_data import load_school_data

BASE_DIR = ROOT / "data" / "interim" / "school"
EXTRACTED_RUN = "run_20261104T090422_148871Z"   # 11/4 추출 완료(LLM 38회). None이면 새로 호출
WORKERS = 4

M, G = "moe_school_manual_v3", "gyeonggi_school_guide_2025"


def rows(doc, page, table, numbers):
    return [f"{doc}:p{page:04d}:t{table:02d}r{n:02d}" for n in numbers]


TARGET_IDS = (rows(M, 38, 1, [1, 2, 3]) + rows(M, 60, 2, [1, 2, 3]) + rows(M, 65, 1, [1, 2, 3, 4])
              + rows(M, 66, 1, [1, 2, 3]) + rows(M, 66, 2, [1, 2]) + rows(M, 66, 3, [1, 2])
              + rows(M, 67, 1, [1, 2]) + rows(M, 67, 2, [1, 2])
              + rows(G, 56, 1, [1, 2, 3]) + rows(G, 165, 2, range(1, 9)) + rows(G, 166, 1, [1, 2])
              + rows(G, 167, 1, [1, 2, 3]) + rows(G, 168, 1, [1]))

books = load_dictionaries()
_, sources_df, blocks_df = load_school_data(ROOT)
all_chunks, _ = build_all_chunks(ROOT, load_manifest(ROOT), books)
sections = {c["chunk_id"]: c["section"] for c in all_chunks}
chunks = build_role_row_chunks(blocks_df, sources_df, TARGET_IDS, sections)
assert len(chunks) == len(TARGET_IDS) == 38 and len({c["chunk_id"] for c in chunks}) == 38
display(pd.DataFrame([{"공식 chunk": c["official_chunk_id"].split(":", 1)[1], "문서": c["document_id"][5:],
                       "PDF": c["pdf_page"], "담당 행": c["role_label"], "표": c["section"][-34:]} for c in chunks]))

,공식 chunk,문서,PDF,담당 행,표
0,p0038:t01r01,school_manual,38,담임교사,쇄 24쪽 · 모의 훈련 시나리오: 대응 2단계 주요 조치사항
1,p0038:t01r02,school_manual,38,보건(담당)교사,쇄 24쪽 · 모의 훈련 시나리오: 대응 2단계 주요 조치사항
2,p0038:t01r03,school_manual,38,교장 및 교감,쇄 24쪽 · 모의 훈련 시나리오: 대응 2단계 주요 조치사항
3,p0060:t02r01,school_manual,60,부장급 교사,쇄 46쪽 · [표 3-9] 대응 제2단계 능동감시 실시 체계
4,p0060:t02r02,school_manual,60,능동감시 대상 학급 담임교사,쇄 46쪽 · [표 3-9] 대응 제2단계 능동감시 실시 체계
5,p0060:t02r03,school_manual,60,보건(담당)교사,쇄 46쪽 · [표 3-9] 대응 제2단계 능동감시 실시 체계
6,p0065:t01r01,school_manual,65,부장급 교사,인쇄 51쪽 · [표 3-11] 능동감시 실시 담당자별 역할
7,p0065:t01r02,school_manual,65,담임교사,인쇄 51쪽 · [표 3-11] 능동감시 실시 담당자별 역할
8,p0065:t01r03,school_manual,65,학년부장,인쇄 51쪽 · [표 3-11] 능동감시 실시 담당자별 역할
9,p0065:t01r04,school_manual,65,보건(담당)교사,인쇄 51쪽 · [표 3-11] 능동감시 실시 담당자별 역할


## LLM 추출

청크마다 한 번(실패하면 한 번 더) 호출한다. 한 청크가 실패해도 나머지는 계속하고 실패는 `run.jsonl`에 남는다.

In [2]:
concept_draft = load_concept_draft(ROOT)
chunks_by_id = {c["chunk_id"]: c for c in chunks}
if EXTRACTED_RUN is None:
    model_name = os.getenv("SCHOOL_EXTRACTION_MODEL", DEFAULT_MODEL)     # 어제와 같은 모델
    model = make_model(model_name)

    def work(chunk):
        return extract_chunk(chunk, concept_draft, model, model_name)

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        outcomes = list(pool.map(work, chunks))        # 입력 순서대로 돌려준다
    runs, candidates = [], []
    for record, found in outcomes:
        runs.append(record)
        candidates += found
    print("상태:", dict(Counter(r["status"] for r in runs)), "/ 후보", len(candidates),
          "/ 토큰", sum((r["token_usage"] or {}).get("total_tokens", 0) for r in runs))
    candidates = annotate_candidates(candidates, chunks_by_id, concept_draft)
    run_dir = save_run(BASE_DIR, chunks, candidates, runs)
else:
    run_dir = BASE_DIR / EXTRACTED_RUN
print("실행 폴더:", run_dir.relative_to(ROOT))

실행 폴더: data\interim\school\run_20261104T090422_148871Z


## 검수 전 자동 점검

사람 검수를 뺀 채로 형식·원문 근거·역할/개념 사전 연결을 돌린 결과다. `auto_reasons`가 있는 후보는 **승인해도 정규화에서 보류된다.**

In [3]:
artifacts = load_artifacts(run_dir)
checks = precheck(artifacts, load_raw_texts(ROOT, artifacts["chunks"]), books)
chunk_label = {c["chunk_id"]: c["official_chunk_id"].split(":", 1)[1] + " " + c["role_label"] for c in chunks}
table = pd.DataFrame([{"#": row["candidate_index"], "행": chunk_label[row["source_chunk_id"]],
                       "stage": row["stage"], "modality": row["modality"], "action": row["action_text"][:46],
                       "역할": ", ".join(row["roles"]), "개념": row["concept"],
                       "보정": "; ".join(n.split(" (")[0] for n in row["corrections"]),
                       "자동 점검": "; ".join(row["auto_reasons"])[:80]}
                      for row, c in zip(checks, artifacts["candidates"], strict=True)])
print("후보", len(table), "/ 자동 점검 통과", int((table["자동 점검"] == "").sum()), "/ 걸림", int((table["자동 점검"] != "").sum()))
print("청크별 후보 수:", dict(Counter(chunk_label[c["source_chunk_id"]] for c in artifacts["candidates"])))
print("단계(보정 후):", dict(Counter(r["stage"] for r in checks)), "/ 보정된 후보:", sum(bool(r["corrections"]) for r in checks))
print("개념:", dict(Counter(str(r["concept"]) for r in checks)))
print("실패한 호출:", [(r["source_chunk_id"], r["status"]) for r in artifacts["failed_runs"]] or "없음")
pd.set_option("display.max_colwidth", 90)
display(table[table["자동 점검"] != ""])

후보 187 / 자동 점검 통과 167 / 걸림 20
청크별 후보 수: {'p0038:t01r01 담임교사': 5, 'p0038:t01r02 보건(담당)교사': 6, 'p0038:t01r03 교장 및 교감': 2, 'p0060:t02r01 부장급 교사': 2, 'p0060:t02r02 능동감시 대상 학급 담임교사': 2, 'p0060:t02r03 보건(담당)교사': 1, 'p0065:t01r01 부장급 교사': 3, 'p0065:t01r02 담임교사': 2, 'p0065:t01r03 학년부장': 1, 'p0065:t01r04 보건(담당)교사': 2, 'p0066:t01r01 보건(담당)교사': 2, 'p0066:t01r02 담임교사': 5, 'p0066:t01r03 상담(담당)교사': 1, 'p0066:t02r01 보건(담당)교사': 2, 'p0066:t02r02 담임교사': 5, 'p0066:t03r01 보건(담당)교사': 3, 'p0066:t03r02 담임교사': 6, 'p0067:t01r01 보건(담당)교사': 3, 'p0067:t01r02 담임교사': 2, 'p0067:t02r01 행정지원팀': 4, 'p0067:t02r02 각 실 담당자': 2, 'p0056:t01r01 담임교사': 5, 'p0056:t01r02 보건(담당)교사': 5, 'p0056:t01r03 교장 및 교감': 2, 'p0165:t02r02 학교장': 2, 'p0165:t02r03 학교장': 6, 'p0165:t02r04 학교장': 2, 'p0165:t02r05 교감': 6, 'p0165:t02r06 교무부장': 10, 'p0165:t02r07 교무부장': 1, 'p0165:t02r08 학년부장': 2, 'p0166:t01r01 담임교사': 21, 'p0166:t01r02 담임교사': 13, 'p0167:t01r01 담임교사': 25, 'p0167:t01r02 담임교사': 2, 'p0167:t01r03 보건교사': 11, 'p0168:t01r01 보건교사': 13}
단계(보정 후):

,#,행,stage,modality,action,역할,개념,보정,자동 점검
2,3,p0038:t01r01 담임교사,RESPONSE_2,UNSPECIFIED,결석·조퇴·지각한 학생의 사유를 확인하고 인플루엔자 의심증상 발생 여부를 조사하여,,ACTIVE_SURVEILLANCE,,표 행 담당(담임교사)이 아닌 역할만 수행자로 추출됨: [] - 행동 문장이 이 행의 담당 행동인지 확인 필요
3,4,p0038:t01r01 담임교사,RESPONSE_2,UNSPECIFIED,(의심)환자 발생 여부를 보건교사에게 통보한다.,,CONTACT_HEALTH_TEACHER,,표 행 담당(담임교사)이 아닌 역할만 수행자로 추출됨: [] - 행동 문장이 이 행의 담당 행동인지 확인 필요
4,5,p0038:t01r01 담임교사,RESPONSE_2,UNSPECIFIED,보건교사로부터 제공받은 예방수칙 및 가정통신문을 교육하고 안내한다.,,HEALTH_EDUCATION,,표 행 담당(담임교사)이 아닌 역할만 수행자로 추출됨: [] - 행동 문장이 이 행의 담당 행동인지 확인 필요
11,12,p0038:t01r03 교장 및 교감,RESPONSE_2,UNSPECIFIED,발생감시팀에게 추가 환자 파악 위한 능동감시 실시를 지시,,ACTIVE_SURVEILLANCE,,역할 연결 실패(COMPOUND): 교장 및 교감 - 한 칸에 여러 역할 - 사람 검수 필요
12,13,p0038:t01r03 교장 및 교감,RESPONSE_2,UNSPECIFIED,그 결과를 보건교사에게 통보 요청,,CONTACT_HEALTH_TEACHER,,역할 연결 실패(COMPOUND): 교장 및 교감 - 한 칸에 여러 역할 - 사람 검수 필요
13,14,p0060:t02r01 부장급 교사,RESPONSE_2,UNSPECIFIED,능동감시를 한다.,INFECTION_MGMT_TEAM(EXAMPLE),ACTIVE_SURVEILLANCE,MANDATORY→UNSPECIFIED,역할 연결 실패(UNREGISTERED): 부장급 교사 - 사전 미등록 역할(승인 필요)
14,15,p0060:t02r01 부장급 교사,RESPONSE_2,UNSPECIFIED,능동감시 결과를 보건(담당)교사에게 통보한다.,INFECTION_MGMT_TEAM(EXAMPLE),CONTACT_HEALTH_TEACHER,MANDATORY→UNSPECIFIED,역할 연결 실패(UNREGISTERED): 부장급 교사 - 사전 미등록 역할(승인 필요)
15,16,p0060:t02r02 능동감시 대상 학급 담임교사,RESPONSE_2,UNSPECIFIED,"결석, 조퇴, 지각한 학생의 사유를 확인하고, 학급 학생들의 증상 유무를 관찰하는",,ACTIVE_SURVEILLANCE,,역할 연결 실패(UNRESOLVED): 능동감시 대상 학급 담임교사 - 사전에 없음
16,17,p0060:t02r02 능동감시 대상 학급 담임교사,RESPONSE_2,UNSPECIFIED,추가 (의심)환자 발생 여부를 보건(담당)교사에게 통보한다.,,CONTACT_HEALTH_TEACHER,,역할 연결 실패(UNRESOLVED): 능동감시 대상 학급 담임교사 - 사전에 없음
18,19,p0065:t01r01 부장급 교사,RESPONSE_3,UNSPECIFIED,능동감시체계 운영에 관한 업무를 지휘하고 발생 감시팀을 총괄한다.,,NEW_CONCEPT,,역할 연결 실패(UNREGISTERED): 부장급 교사 - 사전 미등록 역할(승인 필요)


## 검수 화면

후보마다 원문(제목·머리글·행)과 모델 출력을 보고 판정한다. 행동·개념은 고칠 수 있고, **역할·조건은 고칠 수 없으므로 틀리면 반려**한다.

In [4]:
from preprocess.school.review_ui import create_review_ui

display(create_review_ui(BASE_DIR))